# 05 - XAI Analysis

This notebook adds explainability analysis for the ASD classifier using:
- SHAP (global + local)
- LIME (local)
- PDP (partial dependence)
- Counterfactual-style what-if changes


In [ ]:
import pathlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

ROOT = pathlib.Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'data_csv.csv'
MODEL_PATH = ROOT / 'ml_models' / 'trained_model.pkl'

df = pd.read_csv(DATA_PATH)
pipeline = joblib.load(MODEL_PATH)
feature_names = list(getattr(pipeline, 'feature_names_in_', []))
X = df[feature_names].dropna().copy()
X_sample = X.head(300)
print(type(pipeline))
print('Features:', len(feature_names))
print(feature_names)

## SHAP - Global and Local

In [ ]:
import shap

model = pipeline.named_steps.get('classifier', pipeline)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

sv = shap_values[-1] if isinstance(shap_values, list) else shap_values
if hasattr(sv, 'shape') and len(sv.shape) == 3:
    sv = sv[:, :, -1]

shap.summary_plot(sv, X_sample, plot_type='bar')

In [ ]:
shap.summary_plot(sv, X_sample)

In [ ]:
idx = 0
shap.plots._waterfall.waterfall_legacy(explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value, sv[idx], feature_names=feature_names)

## LIME - Local Explanation

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

class_names = ['NO', 'YES']
lime_explainer = LimeTabularExplainer(
    training_data=X_sample.values,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification'
)

instance = X_sample.iloc[0].values
exp = lime_explainer.explain_instance(instance, pipeline.predict_proba, num_features=8)
exp.as_pyplot_figure()
plt.tight_layout()

## PDP - Partial Dependence

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
PartialDependenceDisplay.from_estimator(pipeline, X_sample, ['Age_Years'], ax=ax[0])
PartialDependenceDisplay.from_estimator(pipeline, X_sample, ['Qchat_10_Score'], ax=ax[1])
plt.tight_layout()

## Counterfactual-style What-If Analysis

In [ ]:
row = X_sample.iloc[0].copy()
base_proba = pipeline.predict_proba(pd.DataFrame([row]))[0][1]
print('Base YES probability:', round(base_proba, 4))

candidates = ['A1', 'A9', 'Age_Years', 'Qchat_10_Score', 'Childhood Autism Rating Scale']
for f in candidates:
    r2 = row.copy()
    if f in ['A1', 'A9']:
        r2[f] = 1 - r2[f]
    elif f == 'Age_Years':
        r2[f] = max(0, r2[f] - 2)
    else:
        r2[f] = max(0, r2[f] - 1)
    p2 = pipeline.predict_proba(pd.DataFrame([r2]))[0][1]
    print(f, '=>', round(p2, 4), '(delta:', round(p2 - base_proba, 4), ')')